# Análise Fuzzy Factory 

**Este conjunto de dados é um banco de dados de comércio eletrônico para Maven Fuzzy Factory, um varejista on-line especializado na venda de ursinhos de pelúcia.**

# Importação dos dados

In [5]:
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f'postgresql+psycopg2://{os.getenv("DB_USER")}:{os.getenv("DB_PASSWORD")}'
    f'@{os.getenv("DB_HOST")}:{os.getenv("DB_PORT")}/db_fuzzy_factory'
)

# Importando arquivos CSVs para o Banco de dados

In [3]:
# Fazendo a leitura dos arquivos e tranformando em DataFrame
df1 = pd.read_csv('fuzzy_factory_csv/maven_fuzzy_factory_data_dictionary.csv')
df2 = pd.read_csv('fuzzy_factory_csv/order_item_refunds.csv')
df3 = pd.read_csv('fuzzy_factory_csv/order_items.csv')
df4 = pd.read_csv('fuzzy_factory_csv/orders.csv')
df5 = pd.read_csv('fuzzy_factory_csv/products.csv')
df6 = pd.read_csv('fuzzy_factory_csv/website_pageviews.csv')
df7 = pd.read_csv('fuzzy_factory_csv/website_sessions.csv')

# Enviando o DataFrame oara o PostgreSQL
df1.to_sql('maven_fuzzy_factory_data_dictionary', engine, if_exists='replace', index=False)
df2.to_sql('order_item_refunds', engine, if_exists='replace', index=False)
df3.to_sql('order_items', engine, if_exists='replace', index=False)
df4.to_sql('orders', engine, if_exists='replace', index=False)
df5.to_sql('products', engine, if_exists='replace', index=False)
df6.to_sql('website_pageviews', engine, if_exists='replace', index=False)
df7.to_sql('website_sessions', engine, if_exists='replace', index=False)

print("Todos os CSVs foram importados com sucesso!")

Todos os CSVs foram importados com sucesso!


# Verificando as tabelas como ficaram

In [4]:
with engine.connect() as conn:
    resultado = conn.execute(text("select table_name from information_schema.tables where table_schema = 'public'"))
    for linha in resultado:
        print(linha[0])

maven_fuzzy_factory_data_dictionary
order_item_refunds
order_items
orders
products
website_pageviews
website_sessions


# Biblioteca do CSV

In [5]:
with engine.connect() as conn:
    query = """

    select *
    from maven_fuzzy_factory_data_dictionary;

    """
    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)

,Table,Field,Description
0,orders,order_id,Unique identifier for each order (PK)
1,orders,created_at,Timestamp when the order was placed
2,orders,website_session_id,Unique identifier for the website session (FK)
3,orders,user_id,Unique identifier for the user (FK)
4,orders,primary_product_id,Unique identifier for the primary product in t...
5,orders,items_purchased,Number of items in the order
6,orders,price_usd,Total price for the items in the order
7,orders,cogs_usd,Cost of goods sold for the items in the order
8,order_items,order_item_id,Unique identifier for each order item (PK)
9,order_items,created_at,Timestamp when the order was placed


# 1. Qual o faturamento da empresa

In [53]:
with engine.connect() as conn:
    query = """

    -- A empresa teve seu maior faturamento em dezembro de 2014, com US$ 214.665,34
    -- É notável que a empresa está em crescimento, porém é importante destacar que
    -- o faturamento só ficou acima da média a partir de fevereiro de 2014, quase 3 anos
    -- após o início das vendas.
    -- No começo, as variações de um mês para o outro eram grandes, depois de 1 ano estabilizou.

    with faturamento_mensal as (
    
    select
        to_char(oi.created_at::date, 'YYYY-MM') as data,
        round(sum(o.price_usd)::numeric, 2) as faturamento_mes_atual
    from order_items oi
    join orders o on o.order_id = oi.order_id
    where oi.created_at is not null 
        and o.price_usd is not null
    group by to_char(oi.created_at::date, 'YYYY-MM')
    order by to_char(oi.created_at::date, 'YYYY-MM') asc

    )

    select 
        data,
        lag(faturamento_mes_atual) over (order by data) as faturamento_mes_anterior,
        faturamento_mes_atual,
        round(
            ((faturamento_mes_atual - lag(faturamento_mes_atual) over (order by data)) 
            / lag(faturamento_mes_atual) over (order by data)) * 100, 2
            ) as variacao_percentual,
        round(avg(faturamento_mes_atual) over(), 2) as media_mes,
        case
            when faturamento_mes_atual > round(avg(faturamento_mes_atual) over(), 2) then 'Acima da Média'
            else 'Abaixo da Média'
        end as classificacao
    from faturamento_mensal
    order by data

    """
    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)

with open('01_faturamento_total.sql', 'w') as f:
    f.write(query)

df.to_csv('01_faturamento_total.csv', index=False)

,data,faturamento_mes_anterior,faturamento_mes_atual,variacao_percentual,media_mes,classificacao
0,2012-03,None,2999.40,None,70994.91,Abaixo da Média
1,2012-04,2999.40,4949.01,65.00,70994.91,Abaixo da Média
2,2012-05,4949.01,5398.92,9.09,70994.91,Abaixo da Média
3,2012-06,5398.92,6998.60,29.63,70994.91,Abaixo da Média
4,2012-07,6998.60,8448.31,20.71,70994.91,Abaixo da Média
5,2012-08,8448.31,11397.72,34.91,70994.91,Abaixo da Média
6,2012-09,11397.72,14347.13,25.88,70994.91,Abaixo da Média
7,2012-10,14347.13,18546.29,29.27,70994.91,Abaixo da Média
8,2012-11,18546.29,30893.82,66.58,70994.91,Abaixo da Média
9,2012-12,30893.82,25294.94,-18.12,70994.91,Abaixo da Média


# 2. Através de qual mídia os produtos mais vendem

In [54]:
with engine.connect() as conn:
    query = """

    -- 65% dos compradores totais da Fuzzy Factory vem pelo Google search, trazendo US$ 1.276.144,89
    -- em faturamento.

    with midia as (

    select 
        case 
            when utm_source = 'gsearch' then 'Google Ads'
            when utm_source = 'bsearch' then 'Bing Ads'
            when utm_source = 'socialbook' then 'Facebook Ads'
            when utm_source is null then 'Tráfego Direto'
            else utm_source
        end as origem_trafego,       
        round(sum(o.price_usd)::numeric, 2) as faturamento
    from orders o
    join website_sessions ws on ws.website_session_id = o.website_session_id
    group by origem_trafego
    )

    select
        origem_trafego,
        faturamento,
        round((faturamento / sum(faturamento) over()) * 100, 2) as percentual_do_total
    from midia
    order by faturamento desc;

    """
    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)

with open('02_midia.sql', 'w') as f:
    f.write(query)

df.to_csv('02_midia.csv', index=False)

,origem_trafego,faturamento,percentual_do_total
0,Google Ads,1276144.89,65.83
1,Tráfego Direto,371433.03,19.16
2,Bing Ads,268672.50,13.86
3,Facebook Ads,22259.33,1.15


# 3. Qual produto traz lucro líquido para a empresa

In [55]:
with engine.connect() as conn:
    query = """

    -- The Original Mr. Fuzzy é lider absoluto nas vendas, com US$ 738.893,00 de lucro líquido.
    -- 3x mais que o segundo lugar, o The Forever Love Bear que faturou US$ 217.350,00.

    select
        p.product_name as produto,
        round(sum(oi.price_usd - oi.cogs_usd)::numeric, 2) as lucro_liquido
    from products p
    join order_items oi on oi.product_id = p.product_id
    group by p.product_name
    order by lucro_liquido desc;

    """
    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)

with open('03_produto_lucro.sql', 'w') as f:
    f.write(query)

df.to_csv('03_produto_lucro.csv', index=False)

,produto,lucro_liquido
0,The Original Mr. Fuzzy,738893.00
1,The Forever Love Bear,217350.00
2,The Birthday Sugar Panda,157027.50
3,The Hudson River Mini bear,102869.00


# 4. Qual produto tem mais vendas em quantidade

In [56]:
with engine.connect() as conn:
    query = """

    -- The Original Mr. Fuzzy lidera também em quantidades, vendendo 24226 produtos.
    -- O segundo lugar, The Forever Love Bear vendeu 5796 quantidades, 
    -- muito abaixo do primeiro lugar.

    select
        p.product_name as produto,
        count(*) as quantidade_vendida
    from products p
    join order_items oi on oi.product_id = p.product_id
    group by p.product_name
    order by quantidade_vendida desc;

    """
    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)

with open('04_venda_quantidade.sql', 'w') as f:
    f.write(query)

df.to_csv('04_venda_quantidade.csv', index=False)

,produto,quantidade_vendida
0,The Original Mr. Fuzzy,24226
1,The Forever Love Bear,5796
2,The Hudson River Mini bear,5018
3,The Birthday Sugar Panda,4985


# 5. O produto que mais traz lucro líquido, é mais vendido através de qual campanha

In [57]:
with engine.connect() as conn:
    query = """

    -- A campanha que o The Original Mr. Fuzzy foi mais vendido é chamada de Campanha Genérica,
    -- Com 17192 quantidades vendidas, e trazendo US$ 524.356,00 de lucro líquido.
    -- Isso significa que a empresa está conquistando clientes que não conheciam a marca.

    select
        p.product_name as produto,
        case 
            when utm_campaign = 'nonbrand' then 'Campanha Genérica'
            when utm_campaign = 'brand' then 'Marca'
            when utm_campaign is null then 'Sem Campanha'
            when utm_campaign = 'desktop_targeted' then 'Segmentada Desktop'
            when utm_campaign = 'pilot' then 'Piloto'
            else utm_campaign
        end as campanha,
        count(*) as quantidade_vendida,
        round(sum(oi.price_usd - oi.cogs_usd)::numeric, 2) as lucro_liquido        
    from orders o
    join website_sessions ws on ws.website_session_id = o.website_session_id
    join order_items oi on oi.order_id = o.order_id
    join products p on p.product_id = oi.product_id
    where p.product_name = 'The Original Mr. Fuzzy'
    group by p.product_name, ws.utm_campaign
    order by quantidade_vendida desc;

    """
    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)

with open('05_produto_campanha.sql', 'w') as f:
    f.write(query)

df.to_csv('05_produto_campanha.csv', index=False)

,produto,campanha,quantidade_vendida,lucro_liquido
0,The Original Mr. Fuzzy,Campanha Genérica,17192,524356.00
1,The Original Mr. Fuzzy,Sem Campanha,4492,137006.00
2,The Original Mr. Fuzzy,Marca,2309,70424.50
3,The Original Mr. Fuzzy,Segmentada Desktop,199,6069.50
4,The Original Mr. Fuzzy,Piloto,34,1037.00


# 6. Qual produto teve mais reembolso

In [58]:
with engine.connect() as conn:
    query = """

    -- Por ser o produto mais vendido, The Original Mr. Fuzzy também é o produto que mais teve reembolso
    -- US$ 69.361,92 foi reembolsado.

    select 
        p.product_name as produto,
        round(sum(oir.refund_amount_usd)::numeric, 2) as total_reembolso
    from order_item_refunds oir
    join order_items oi on oi.order_id = oir.order_id
    join products p on p.product_id = oi.product_id
    group by p.product_name
    order by total_reembolso desc
    limit 10;
    
    """
    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)

with open('06_produto_reembolso.sql', 'w') as f:
    f.write(query)

df.to_csv('06_produto_reembolso.csv', index=False)

,produto,total_reembolso
0,The Original Mr. Fuzzy,69361.92
1,The Birthday Sugar Panda,19201.89
2,The Hudson River Mini bear,12367.26
3,The Forever Love Bear,11457.91


# 7. Através de qual dispositivo existe mais vendas (mobile/desktop)

In [59]:
with engine.connect() as conn:
    query = """

    -- As vendas do The Original Mr. Fuzzy lideram através do Desktop, 
    -- faturando US$ 638.822,50 e com 20945 em quantidades vendidas.
    -- Já pelo Mobile o mesmo produto também é o mais vendido, porém pelo
    -- faturamento de US$ 100.070,50.

    select *
    from (

    select
        ws.device_type as dispositivo,
        p.product_name as produto,
        count(*) as quantidade_vendida,
        round(sum(oi.price_usd - oi.cogs_usd)::numeric, 2) as lucro_liquido,
        rank() over (partition by ws.device_type order by round(sum(oi.price_usd - oi.cogs_usd)::numeric, 2) desc) as ranking
    from orders o
    join website_sessions ws on ws.website_session_id = o.website_session_id
    join order_items oi on oi.order_id = o.order_id
    join products p on p.product_id = oi.product_id
    group by p.product_name, ws.device_type
    limit 60

    )

    where ranking <= 3
        
    """
    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)

with open('07_dispositivo.sql', 'w') as f:
    f.write(query)

df.to_csv('07_dispositivo.csv', index=False)

,dispositivo,produto,quantidade_vendida,lucro_liquido,ranking
0,desktop,The Original Mr. Fuzzy,20945,638822.50,1
1,desktop,The Forever Love Bear,4847,181762.50,2
2,desktop,The Birthday Sugar Panda,4332,136458.00,3
3,mobile,The Original Mr. Fuzzy,3281,100070.50,1
4,mobile,The Forever Love Bear,949,35587.50,2
5,mobile,The Birthday Sugar Panda,653,20569.50,3
